# 📋 SMD Dataset — Complete Feature Table

**Dataset:** Server Machine Dataset (SMD)  
**Source:** [NetManAIOps/OmniAnomaly](https://github.com/NetManAIOps/OmniAnomaly)  
**Purpose:** Generate the perfect dataset overview table with all 38 sensor features and all machine-level stats.

---

### What this notebook produces
| Table | Description |
|-------|-------------|
| **Table 1** | Dataset-level summary card |
| **Table 2** | Per-machine statistics (all 27 machines) |
| **Table 3** | Per-feature (sensor) statistics — train set |
| **Table 4** | Per-feature statistics — test set (with anomaly context) |
| **Table 5** | Feature data-type / range / missing value catalogue |

In [1]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add project root (training/) to path so src.* imports work
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

from src.data_loader import (
    load_smd_machine, get_dataset_info, load_as_dataframe,
    NUM_FEATURES, MACHINE_IDS, DEFAULT_DATA_DIR
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_rows', 60)

print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')
print(f'Machines in SMD : {len(MACHINE_IDS)}')
print(f'Features per machine : {NUM_FEATURES}')
print(f'Data directory : {DEFAULT_DATA_DIR}')

NumPy  : 2.4.6
Pandas : 3.0.5
Machines in SMD : 27
Features per machine : 38
Data directory : d:\ojt-project\data\raw\ServerMachineDataset


---
## Table 1 — Dataset-Level Summary

In [2]:
# ── Table 1: Dataset-Level Summary Card ───────────────────────────────────────
info = get_dataset_info()

if info['status'] != 'available':
    raise RuntimeError(
        'Dataset not found. Run scripts/download_dataset.py first.'
    )

machines_df_raw = pd.DataFrame(info['machines'])

total_train_rows = machines_df_raw['n_train'].sum()
total_test_rows  = machines_df_raw['n_test'].sum()
total_anomaly    = machines_df_raw['anomaly_count'].sum()
overall_anom_ratio = total_anomaly / total_test_rows

summary_card = pd.DataFrame([
    ('Name',                   'Server Machine Dataset (SMD)'),
    ('Source',                 'https://github.com/NetManAIOps/OmniAnomaly'),
    ('Task',                   'Unsupervised Time-Series Anomaly Detection'),
    ('Number of machines',     len(info['machines'])),
    ('Features per machine',   NUM_FEATURES),
    ('Feature type',           'Continuous (normalised 0-1 by original authors)'),
    ('Label type',             'Binary  0 = Normal  |  1 = Anomaly'),
    ('Total TRAIN rows',       f'{total_train_rows:,}'),
    ('Total TEST rows',        f'{total_test_rows:,}'),
    ('Total anomalous rows',   f'{total_anomaly:,}  ({overall_anom_ratio*100:.2f} %)'),
    ('Sampling frequency',     '~1 minute (implied by original paper)'),
    ('Missing values',         'None — dataset is complete'),
    ('File format',            'CSV-like .txt  (comma-separated, no header)'),
    ('Train/Test split',       'Temporal — train = normal only, test = mixed'),
], columns=['Property', 'Value'])

print('=' * 60)
print('  TABLE 1: SMD Dataset-Level Summary')
print('=' * 60)
display(summary_card.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', '#f8fafc'),
                                      ('font-size', '13px'), ('padding', '8px 14px')]},
        {'selector': 'td', 'props': [('padding', '6px 14px'), ('font-size', '12px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f1f5f9')]},
    ])
    .hide(axis='index')
)

  TABLE 1: SMD Dataset-Level Summary


Property,Value
Name,Server Machine Dataset (SMD)
Source,https://github.com/NetManAIOps/OmniAnomaly
Task,Unsupervised Time-Series Anomaly Detection
Number of machines,27
Features per machine,38
Feature type,Continuous (normalised 0-1 by original authors)
Label type,Binary 0 = Normal | 1 = Anomaly
Total TRAIN rows,"679,683"
Total TEST rows,"679,698"
Total anomalous rows,"27,689 (4.07 %)"


---
## Table 2 — Per-Machine Statistics (all 27 machines)

In [3]:
# ── Table 2: Per-Machine Stats ─────────────────────────────────────────────────
rows = []
for m in info['machines']:
    mid  = m['machine_id']
    grp  = mid.split('-')[1]
    rows.append({
        'Machine ID':        mid,
        'Group':             f'Group {grp}',
        'Train Rows':        m['n_train'],
        'Test Rows':         m['n_test'],
        'Total Rows':        m['n_train'] + m['n_test'],
        'Anomaly Count':     m['anomaly_count'],
        'Anomaly Ratio (%)': round(m['anomaly_ratio'] * 100, 3),
        'Normal Count':      m['n_test'] - m['anomaly_count'],
        'Normal Ratio (%)':  round((1 - m['anomaly_ratio']) * 100, 3),
        'Features':          NUM_FEATURES,
    })

machines_table = pd.DataFrame(rows)

totals = {
    'Machine ID': 'TOTAL',
    'Group': '—',
    'Train Rows': machines_table['Train Rows'].sum(),
    'Test Rows':  machines_table['Test Rows'].sum(),
    'Total Rows': machines_table['Total Rows'].sum(),
    'Anomaly Count': machines_table['Anomaly Count'].sum(),
    'Anomaly Ratio (%)': round(machines_table['Anomaly Count'].sum() /
                               machines_table['Test Rows'].sum() * 100, 3),
    'Normal Count': machines_table['Normal Count'].sum(),
    'Normal Ratio (%)': round(machines_table['Normal Count'].sum() /
                              machines_table['Test Rows'].sum() * 100, 3),
    'Features': NUM_FEATURES,
}
machines_table = pd.concat(
    [machines_table, pd.DataFrame([totals])], ignore_index=True
)

print('=' * 60)
print('  TABLE 2: Per-Machine Statistics')
print('=' * 60)
display(machines_table.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', '#f8fafc'),
                                      ('font-size', '12px'), ('padding', '8px 12px'),
                                      ('text-align', 'center')]},
        {'selector': 'td', 'props': [('padding', '5px 12px'), ('font-size', '11px'),
                                      ('text-align', 'center')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f1f5f9')]},
        {'selector': 'tr:last-child td', 'props': [('font-weight', 'bold'),
                                                    ('background-color', '#dbeafe')]},
    ])
    .format({
        'Train Rows':        '{:,}',
        'Test Rows':         '{:,}',
        'Total Rows':        '{:,}',
        'Anomaly Count':     '{:,}',
        'Normal Count':      '{:,}',
        'Anomaly Ratio (%)': '{:.3f}',
        'Normal Ratio (%)':  '{:.3f}',
    })
    .hide(axis='index')
    .bar(subset=['Anomaly Ratio (%)'], color='#fca5a5', vmin=0)
)

  TABLE 2: Per-Machine Statistics


Machine ID,Group,Train Rows,Test Rows,Total Rows,Anomaly Count,Anomaly Ratio (%),Normal Count,Normal Ratio (%),Features
machine-1-1,Group 1,"28,479","28,479","56,958","2,694",9.460,"25,785",90.540,38
machine-1-2,Group 1,"23,694","23,694","47,388",542,2.287,"23,152",97.713,38
machine-1-3,Group 1,"23,702","23,703","47,405",817,3.447,"22,886",96.553,38
machine-1-4,Group 1,"23,706","23,707","47,413",720,3.037,"22,987",96.963,38
machine-1-5,Group 1,"23,705","23,706","47,411",100,0.422,"23,606",99.578,38
machine-1-6,Group 1,"23,688","23,689","47,377","3,708",15.653,"19,981",84.347,38
machine-1-7,Group 1,"23,697","23,697","47,394","2,398",10.119,"21,299",89.881,38
machine-1-8,Group 1,"23,698","23,699","47,397",763,3.220,"22,936",96.780,38
machine-2-1,Group 2,"23,693","23,694","47,387","1,170",4.938,"22,524",95.062,38
machine-2-2,Group 2,"23,699","23,700","47,399","2,833",11.954,"20,867",88.046,38


---
## Table 3 — Per-Feature Statistics: Train Set (machine-1-1)

In [4]:
# ── Table 3: Per-Feature Stats — Train Set ────────────────────────────────────
MACHINE_ID = 'machine-1-1'
data = load_smd_machine(MACHINE_ID)
train_df = load_as_dataframe(MACHINE_ID, split='train')
test_df  = load_as_dataframe(MACHINE_ID, split='test')

sensor_cols = [f'sensor_{i}' for i in range(NUM_FEATURES)]

train_stats = train_df[sensor_cols].describe(percentiles=[.05, .25, .50, .75, .95]).T
train_stats.index.name = 'Feature'
train_stats = train_stats.reset_index()

train_stats['Feature Index'] = range(NUM_FEATURES)
train_stats['Variance']      = train_df[sensor_cols].var().values
train_stats['IQR']           = (train_df[sensor_cols].quantile(0.75) -
                                 train_df[sensor_cols].quantile(0.25)).values
train_stats['Skewness']      = train_df[sensor_cols].skew().values
train_stats['Kurtosis']      = train_df[sensor_cols].kurtosis().values
train_stats['Zero Ratio (%)']= (train_df[sensor_cols] == 0).mean().values * 100
train_stats['Missing']       = train_df[sensor_cols].isnull().sum().values

train_stats = train_stats.rename(columns={
    'count': 'Count',
    'mean':  'Mean',
    'std':   'Std Dev',
    'min':   'Min',
    '5%':    'P5',
    '25%':   'P25 (Q1)',
    '50%':   'Median',
    '75%':   'P75 (Q3)',
    '95%':   'P95',
    'max':   'Max',
})

col_order = [
    'Feature Index', 'Feature', 'Count', 'Missing', 'Zero Ratio (%)',
    'Mean', 'Std Dev', 'Variance', 'Min', 'P5', 'P25 (Q1)',
    'Median', 'P75 (Q3)', 'P95', 'Max', 'IQR', 'Skewness', 'Kurtosis'
]
train_stats = train_stats[col_order]

print('=' * 60)
print(f'  TABLE 3: Per-Feature Statistics — TRAIN SET ({MACHINE_ID})')
print(f'  Rows: {data["n_train"]:,}   Features: {NUM_FEATURES}')
print('=' * 60)
display(train_stats.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1e3a5f'), ('color', '#f8fafc'),
                                      ('font-size', '11px'), ('padding', '7px 10px'),
                                      ('text-align', 'center'), ('white-space', 'nowrap')]},
        {'selector': 'td', 'props': [('padding', '4px 10px'), ('font-size', '11px'),
                                      ('text-align', 'right')]},
        {'selector': 'td:nth-child(2)', 'props': [('font-weight', 'bold'),
                                                   ('color', '#1e40af'),
                                                   ('text-align', 'left')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f0f9ff')]},
    ])
    .format({
        'Count':          '{:,.0f}',
        'Missing':        '{:,.0f}',
        'Zero Ratio (%)': '{:.2f}',
        'Mean':           '{:.6f}',
        'Std Dev':        '{:.6f}',
        'Variance':       '{:.6f}',
        'Min':            '{:.6f}',
        'P5':             '{:.6f}',
        'P25 (Q1)':       '{:.6f}',
        'Median':         '{:.6f}',
        'P75 (Q3)':       '{:.6f}',
        'P95':            '{:.6f}',
        'Max':            '{:.6f}',
        'IQR':            '{:.6f}',
        'Skewness':       '{:.4f}',
        'Kurtosis':       '{:.4f}',
    })
    .hide(axis='index')
    .bar(subset=['Std Dev'], color='#93c5fd', vmin=0)
    .bar(subset=['Mean'], color='#6ee7b7', vmin=0)
)

  TABLE 3: Per-Feature Statistics — TRAIN SET (machine-1-1)
  Rows: 28,479   Features: 38


Feature Index,Feature,Count,Missing,Zero Ratio (%),Mean,Std Dev,Variance,Min,P5,P25 (Q1),Median,P75 (Q3),P95,Max,IQR,Skewness,Kurtosis
0,sensor_0,"28,479",0,0.08,0.064195,0.056685,0.003213,0.000000,0.010753,0.032258,0.043011,0.086022,0.139785,0.494624,0.053764,3.9085,22.5851
1,sensor_1,"28,479",0,0.01,0.056882,0.042516,0.001808,0.000000,0.011653,0.027542,0.045551,0.075212,0.140890,0.421610,0.047670,1.7152,4.1656
2,sensor_2,"28,479",0,0.00,0.053549,0.037695,0.001421,0.000000,0.014493,0.027871,0.042363,0.070234,0.127090,0.278707,0.042363,1.7034,3.8763
3,sensor_3,"28,479",0,0.00,0.050188,0.037443,0.001402,0.000000,0.012776,0.024390,0.037166,0.067364,0.123113,0.250871,0.042974,1.7359,3.8656
4,sensor_4,"28,479",0,100.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.0000
5,sensor_5,"28,479",0,0.00,0.913552,0.037972,0.001442,0.000000,0.900000,0.907692,0.912821,0.920513,0.938462,1.000000,0.012821,-14.6825,263.2425
6,sensor_6,"28,479",0,0.00,0.262274,0.066661,0.004444,0.000000,0.114878,0.251412,0.274011,0.311676,0.338983,0.360640,0.060264,-1.4989,2.0665
7,sensor_7,"28,479",0,100.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.0000
8,sensor_8,"28,479",0,0.00,0.020346,0.009982,0.000100,0.000000,0.004542,0.012347,0.021005,0.027959,0.035481,0.118081,0.015612,0.1258,-0.2494
9,sensor_9,"28,479",0,52.95,0.000159,0.001287,0.000002,0.000000,0.000000,0.000000,0.000000,0.000122,0.000366,0.140435,0.000122,68.4292,6132.3922


---
## Table 4 — Per-Feature Statistics: Test Set (Normal vs Anomalous)

In [5]:
# ── Table 4: Feature Stats — Normal vs Anomalous (Test Set) ──────────────────
normal_df  = test_df[test_df['label'] == 0][sensor_cols]
anomaly_df = test_df[test_df['label'] == 1][sensor_cols]

feat_rows = []
for i, col in enumerate(sensor_cols):
    norm_vals = normal_df[col]
    anom_vals = anomaly_df[col]
    all_vals  = test_df[col]
    feat_rows.append({
        'Idx':              i,
        'Feature':          col,
        'Test Mean':        all_vals.mean(),
        'Test Std':         all_vals.std(),
        'Normal Mean':      norm_vals.mean(),
        'Normal Std':       norm_vals.std(),
        'Normal Min':       norm_vals.min(),
        'Normal Max':       norm_vals.max(),
        'Anomaly Mean':     anom_vals.mean() if len(anom_vals) else float('nan'),
        'Anomaly Std':      anom_vals.std()  if len(anom_vals) else float('nan'),
        'Anomaly Min':      anom_vals.min()  if len(anom_vals) else float('nan'),
        'Anomaly Max':      anom_vals.max()  if len(anom_vals) else float('nan'),
        'Delta Mean':       (
            anom_vals.mean() - norm_vals.mean()
            if len(anom_vals) else float('nan')
        ),
        'Rel Shift (%)': (
            abs(anom_vals.mean() - norm_vals.mean()) /
            max(abs(norm_vals.mean()), 1e-9) * 100
            if len(anom_vals) else float('nan')
        ),
    })

feat_table = pd.DataFrame(feat_rows)
n_normal  = (test_df['label'] == 0).sum()
n_anomaly = (test_df['label'] == 1).sum()

print('=' * 60)
print(f'  TABLE 4: Feature Stats — Test Set Normal vs Anomalous ({MACHINE_ID})')
print(f'  Normal rows: {n_normal:,}   Anomalous rows: {n_anomaly:,}')
print('=' * 60)
display(feat_table.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1e3a5f'), ('color', '#f8fafc'),
                                      ('font-size', '11px'), ('padding', '7px 10px'),
                                      ('text-align', 'center'), ('white-space', 'nowrap')]},
        {'selector': 'td', 'props': [('padding', '4px 10px'), ('font-size', '11px'),
                                      ('text-align', 'right')]},
        {'selector': 'td:nth-child(2)', 'props': [('font-weight', 'bold'),
                                                   ('color', '#1e40af'),
                                                   ('text-align', 'left')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#fff7ed')]},
    ])
    .format({
        'Test Mean':      '{:.6f}',
        'Test Std':       '{:.6f}',
        'Normal Mean':    '{:.6f}',
        'Normal Std':     '{:.6f}',
        'Normal Min':     '{:.6f}',
        'Normal Max':     '{:.6f}',
        'Anomaly Mean':   '{:.6f}',
        'Anomaly Std':    '{:.6f}',
        'Anomaly Min':    '{:.6f}',
        'Anomaly Max':    '{:.6f}',
        'Delta Mean':     '{:+.6f}',
        'Rel Shift (%)':  '{:.2f}',
    }, na_rep='—')
    .hide(axis='index')
    .background_gradient(
        subset=['Rel Shift (%)'], cmap='Oranges', vmin=0
    )
)

  TABLE 4: Feature Stats — Test Set Normal vs Anomalous (machine-1-1)
  Normal rows: 25,785   Anomalous rows: 2,694


Idx,Feature,Test Mean,Test Std,Normal Mean,Normal Std,Normal Min,Normal Max,Anomaly Mean,Anomaly Std,Anomaly Min,Anomaly Max,Delta Mean,Rel Shift (%)
0,sensor_0,0.145166,0.136204,0.114466,0.090062,0.010753,0.591398,0.439012,0.152045,0.053763,1.000000,+0.324547,283.53
1,sensor_1,0.108322,0.102157,0.098164,0.077606,0.000000,0.679025,0.205551,0.205547,0.005297,1.000000,+0.107387,109.40
2,sensor_2,0.107874,0.102078,0.097395,0.075374,0.002230,0.523969,0.208171,0.211377,0.016722,1.000000,+0.110776,113.74
3,sensor_3,0.106855,0.103836,0.096657,0.077637,0.001161,0.537747,0.204464,0.213965,0.025552,1.000000,+0.107807,111.54
4,sensor_4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,+0.000000,0.00
5,sensor_5,0.915816,0.030684,0.914651,0.028969,0.107692,1.000000,0.926964,0.042241,0.456410,1.000000,+0.012313,1.35
6,sensor_6,0.254842,0.072073,0.248958,0.067144,0.005650,0.500000,0.311156,0.090893,0.175141,1.000000,+0.062198,24.98
7,sensor_7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,+0.000000,0.00
8,sensor_8,0.023751,0.019688,0.022900,0.012801,0.000000,0.483963,0.031895,0.049563,0.000426,1.000000,+0.008995,39.28
9,sensor_9,0.001014,0.015724,0.000168,0.001131,0.000000,0.078083,0.009108,0.050300,0.000000,1.000000,+0.008940,5327.37


---
## Table 5 — Feature Catalogue (Type / Range / Role)

In [6]:
# ── Table 5: Feature Catalogue ────────────────────────────────────────────────
SMD_FEATURE_DESCRIPTIONS = [
    'CPU utilisation',
    'Network In (bytes/s)',
    'Network Out (bytes/s)',
    'Network Total (bytes/s)',
    'NIC Error Rate',
    'Disk Read Rate',
    'Disk Write Rate',
    'Disk Busy Rate',
    'Memory Used Ratio',
    'Memory Free Ratio',
    'Retrans Rate',
    'Network Droprate In',
    'Network Droprate Out',
    'TCP Conn Count',
    'UDP Conn Count',
    'TCP/UDP Ratio',
    'Packet Loss Rate',
    'Server Load 1-min',
    'Server Load 5-min',
    'Server Load 15-min',
    'Process Count',
    'Zombie Process Count',
    'File Descriptor Usage',
    'Disk Space Used (%)',
    'iNode Usage (%)',
    'Page Fault Rate',
    'Swap Usage',
    'Context Switch Rate',
    'Interrupt Rate',
    'Soft IRQ Rate',
    'System Call Rate',
    'User CPU Time',
    'System CPU Time',
    'IO Wait CPU Time',
    'Steal CPU Time',
    'Idle CPU Time',
    'RSS Memory',
    'VMS Memory',
]

catalogue_rows = []
for i in range(NUM_FEATURES):
    col  = f'sensor_{i}'
    t_vals = train_df[col]
    catalogue_rows.append({
        'Index':          i,
        'Column Name':    col,
        'Description':    SMD_FEATURE_DESCRIPTIONS[i],
        'Data Type':      'float32',
        'Value Range':    '[0, 1]',
        'Observed Min':   round(t_vals.min(), 6),
        'Observed Max':   round(t_vals.max(), 6),
        'Train Mean':     round(t_vals.mean(), 6),
        'Train Std':      round(t_vals.std(), 6),
        'Zero Ratio (%)': round((t_vals == 0).mean() * 100, 2),
        'Missing':        int(t_vals.isnull().sum()),
        'Role':           'Input Feature',
    })

catalogue_rows.append({
    'Index':          38,
    'Column Name':    'label',
    'Description':    'Anomaly label (TEST split only)',
    'Data Type':      'int32',
    'Value Range':    '{0, 1}',
    'Observed Min':   0,
    'Observed Max':   1,
    'Train Mean':     float('nan'),
    'Train Std':      float('nan'),
    'Zero Ratio (%)': float('nan'),
    'Missing':        0,
    'Role':           'Target Label',
})

catalogue_df = pd.DataFrame(catalogue_rows)

print('=' * 60)
print('  TABLE 5: Complete Feature Catalogue (39 columns incl. label)')
print('=' * 60)
display(catalogue_df.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#14532d'), ('color', '#f0fdf4'),
                                      ('font-size', '11px'), ('padding', '7px 10px'),
                                      ('text-align', 'center'), ('white-space', 'nowrap')]},
        {'selector': 'td', 'props': [('padding', '4px 10px'), ('font-size', '11px')]},
        {'selector': 'td:nth-child(2)', 'props': [('font-family', 'monospace'),
                                                   ('color', '#065f46')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f0fdf4')]},
        {'selector': 'tr:last-child td', 'props': [('font-weight', 'bold'),
                                                    ('background-color', '#dcfce7')]},
    ])
    .format({
        'Observed Min':   '{:.6f}',
        'Observed Max':   '{:.6f}',
        'Train Mean':     '{:.6f}',
        'Train Std':      '{:.6f}',
        'Zero Ratio (%)': '{:.2f}',
    }, na_rep='—')
    .hide(axis='index')
    .apply(lambda col: ['background-color: #fef9c3' if v == 'Target Label'
                        else '' for v in col], subset=['Role'])
)

  TABLE 5: Complete Feature Catalogue (39 columns incl. label)


Index,Column Name,Description,Data Type,Value Range,Observed Min,Observed Max,Train Mean,Train Std,Zero Ratio (%),Missing,Role
0,sensor_0,CPU utilisation,float32,"[0, 1]",0.000000,0.494624,0.064195,0.056685,0.08,0,Input Feature
1,sensor_1,Network In (bytes/s),float32,"[0, 1]",0.000000,0.421610,0.056882,0.042516,0.01,0,Input Feature
2,sensor_2,Network Out (bytes/s),float32,"[0, 1]",0.000000,0.278707,0.053549,0.037695,0.00,0,Input Feature
3,sensor_3,Network Total (bytes/s),float32,"[0, 1]",0.000000,0.250871,0.050188,0.037443,0.00,0,Input Feature
4,sensor_4,NIC Error Rate,float32,"[0, 1]",0.000000,0.000000,0.000000,0.000000,100.00,0,Input Feature
5,sensor_5,Disk Read Rate,float32,"[0, 1]",0.000000,1.000000,0.913552,0.037972,0.00,0,Input Feature
6,sensor_6,Disk Write Rate,float32,"[0, 1]",0.000000,0.360640,0.262274,0.066661,0.00,0,Input Feature
7,sensor_7,Disk Busy Rate,float32,"[0, 1]",0.000000,0.000000,0.000000,0.000000,100.00,0,Input Feature
8,sensor_8,Memory Used Ratio,float32,"[0, 1]",0.000000,0.118081,0.020346,0.009982,0.00,0,Input Feature
9,sensor_9,Memory Free Ratio,float32,"[0, 1]",0.000000,0.140435,0.000159,0.001287,52.95,0,Input Feature


---
## Export Tables to CSV

In [7]:
# ── Export all tables to CSV ──────────────────────────────────────────────────
OUT_DIR = Path('..') / 'artifacts' / 'dataset_tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_card.to_csv(OUT_DIR / 'table1_dataset_summary.csv',     index=False)
machines_table.to_csv(OUT_DIR / 'table2_per_machine_stats.csv', index=False)
train_stats.to_csv(OUT_DIR / 'table3_train_feature_stats.csv',  index=False)
feat_table.to_csv(OUT_DIR / 'table4_test_normal_vs_anomaly.csv', index=False)
catalogue_df.to_csv(OUT_DIR / 'table5_feature_catalogue.csv',    index=False)

print('All tables exported to:', OUT_DIR.resolve())
for f in sorted(OUT_DIR.iterdir()):
    print(f'   {f.name}  ({f.stat().st_size / 1024:.1f} KB)')

All tables exported to: D:\ojt-project\training\artifacts\dataset_tables
   table1_dataset_summary.csv  (0.6 KB)
   table2_per_machine_stats.csv  (1.9 KB)
   table3_train_feature_stats.csv  (7.1 KB)
   table4_test_normal_vs_anomaly.csv  (6.7 KB)
   table5_feature_catalogue.csv  (3.8 KB)


---
## Quick Sanity Check

In [8]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
print('=== Sanity Checks ===')
print(f'  Machines loaded            : {len(info["machines"])} / {len(MACHINE_IDS)}')
print(f'  Features per machine       : {NUM_FEATURES}')
print(f'  Train shape (machine-1-1)  : {data["train"].shape}')
print(f'  Test shape  (machine-1-1)  : {data["test"].shape}')
print(f'  Labels shape               : {data["test_labels"].shape}')
print(f'  Missing values (train)     : {train_df[sensor_cols].isnull().sum().sum()}')
print(f'  Missing values (test)      : {test_df[sensor_cols].isnull().sum().sum()}')
print(f'  Inf values (train)         : {np.isinf(train_df[sensor_cols].values).sum()}')
print(f'  Inf values (test)          : {np.isinf(test_df[sensor_cols].values).sum()}')
print(f'  Label unique values        : {sorted(test_df["label"].unique())}')
print(f'  Anomaly ratio (test)       : {data["anomaly_ratio"]*100:.3f} %')
print()
print('Dataset integrity verified — ready for modelling.')

=== Sanity Checks ===
  Machines loaded            : 27 / 27
  Features per machine       : 38
  Train shape (machine-1-1)  : (28479, 38)
  Test shape  (machine-1-1)  : (28479, 38)
  Labels shape               : (28479,)
  Missing values (train)     : 0
  Missing values (test)      : 0
  Inf values (train)         : 0
  Inf values (test)          : 0
  Label unique values        : [np.int64(0), np.int64(1)]
  Anomaly ratio (test)       : 9.460 %

Dataset integrity verified — ready for modelling.
